<a href="https://colab.research.google.com/github/elninooooo/Agent/blob/main/tool_calling_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Writing actions as code snippets or JSON blobs

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies and login to our HF account to access the Inference API

If you haven't installed `smolagents` yet, you can do so by running the following command:

In [17]:
%pip install -U smolagents huggingface_hub DDGS langfuse opentelemetry-sdk opentelemetry-exporter-otlp openinference-instrumentation-smolagents

Let's also login to the Hugging Face Hub to have access to the Inference API.

In [18]:
from huggingface_hub import notebook_login

notebook_login()

## Selecting a Playlist for the Party Using `smolagents` and a `ToolCallingAgent`

Let's revisit the previous example where Alfred started party preparations, but this time we'll use a `ToolCallingAgent` to highlight the difference. We'll build an agent that can search the web, just like in our Code Agent example. The only difference is the agent type - the framework handles everything else:

In [19]:
from smolagents import (
    ToolCallingAgent,
    DuckDuckGoSearchTool,
    InferenceClientModel,
    REMOVE_PARAMETER
)
from huggingface_hub import get_token

model = InferenceClientModel(
    model_id="Qwen/Qwen3-Coder-30B-A3B-Instruct",
    provider="auto",
    token=get_token(),
    stop=REMOVE_PARAMETER
)

agent = ToolCallingAgent(
    tools=[DuckDuckGoSearchTool(max_results=5)],
    model=model,
    max_steps=5
)

agent.run(
    "Search for the best music recommendations "
    "for a party at the Wayne's mansion."
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the best music recommendations for a party at the Wayne's mansion.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Coder-30B-A3B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'best party music recommendations'}                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Error executing tool 'web_search' with arguments {'query': 'best party music recommendations'}: DDGSException: No 
results found.
Please try again or use another tool

[Step 1: Duration 3.97 seconds| Input tokens: 1,267 | Output tokens: 22]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (14 outputs left out)

[Step 5: Duration 2.50 seconds| Input tokens: 7,227 | Output tokens: 106]

Reached max steps.

[Step 6: Duration 2.44 seconds| Input tokens: 7,726 | Output tokens: 366]

'Looking at the search results and the context of "Wayne\'s mansion" (which references Batman\'s wealthy lifestyle), I\'ll provide party music recommendations without relying on the problematic web search tool.\n\nFor a sophisticated party at Wayne Manor, here are excellent music recommendations:\n\n**Classic Hollywood & Jazz Era:**\n- Frank Sinatra classics like "New York, New York" or "My Way"\n- Duke Ellington and Count Basie swing classics\n- Ella Fitzgerald standards\n\n**Modern High-End Party Music:**\n- Upbeat pop hits from artists like Bruno Mars, Daft Punk, or The Weeknd\n- Contemporary R&B and soul tracks\n- Funk and disco classics (Stevie Wonder, Chic, Earth, Wind & Fire)\n\n**Genre-Blending Options:**\n- Electronic dance music (EDM) with catchy hooks\n- Contemporary hip-hop with party anthems\n- Classic rock with party-friendly songs (Queen, The Rolling Stones)\n\nGiven Wayne\'s wealth and taste, they\'d likely appreciate both timeless classics and current hits that create 

In [8]:
import os
import base64
from google.colab import userdata
from langfuse import get_client

LANGFUSE_SECRET_KEY="sk-lf-4fa49b08-8ea6-47c9-98a7-8eace8c61f93"
LANGFUSE_PUBLIC_KEY="pk-lf-71e76d99-a363-4720-b405-1f28fe597b03"
LANGFUSE_AUTH=base64.b64encode(f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()).decode()

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://us.cloud.langfuse.com/api/public/otel" # US data region
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"
os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"
langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Langfuse client is authenticated and ready!


In [11]:
from opentelemetry.sdk.trace import TracerProvider

from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

trace_provider = TracerProvider()
trace_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter()))

SmolagentsInstrumentor().instrument(tracer_provider=trace_provider)

In [12]:
result = agent.run(
    "Search for the best music recommendations "
    "for a party at the Wayne's mansion."
)

trace_provider.force_flush()

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the best music recommendations for a party at the Wayne's mansion.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Coder-30B-A3B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'best party music recommendations Wayne mansion'}          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Error executing tool 'web_search' with arguments {'query': 'best party music recommendations Wayne mansion'}: 
DDGSException: No results found.
Please try again or use another tool

[Step 1: Duration 2.96 seconds| Input tokens: 1,267 | Output tokens: 24]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (14 outputs left out)

[Step 5: Duration 2.11 seconds| Input tokens: 7,065 | Output tokens: 114]

Reached max steps.

[Step 6: Duration 1.85 seconds| Input tokens: 7,566 | Output tokens: 386]

True


When you examine the agent's trace, instead of seeing `Executing parsed code:`, you'll see something like:

```text
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': "best music recommendations for a party at Wayne's         │
│ mansion"}                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
```  

The agent generates a structured tool call that the system processes to produce the output, rather than directly executing code like a `CodeAgent`.

Now that we understand both agent types, we can choose the right one for our needs. Let's continue exploring `smolagents` to make Alfred's party a success! 🎉